# 07 — Forecasting Engine and Inventory Optimization

**Dataset limitation:** UCI has sales only — no stock-on-hand, lead time or stockout history.
Current stock, lead time, service level, MOQ, pack size and capacity below are **example user inputs**, clearly labelled.

Policy (periodic review, order-up-to):
* Safety stock `SS = Z × σ_daily × √L`
* Reorder point `ROP = d × L + SS`
* Target stock `T = d × (L + R) + SS` with review period R = 7 days
* Order `Q = max(0, T − stock)`, then MOQ → pack size → capacity

> **Colab:** run the setup cell; it asks for the project ZIP the first time. Put `Online Retail.xlsx` in `data/raw/` (or run notebook 01 first, which uploads it).

In [1]:
import sys, os, zipfile, io
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
ROOT = Path.cwd()
for c in (ROOT, ROOT.parent, ROOT / "AI-Sales-Forecasting-Inventory"):
    if (c / "ml").is_dir():
        ROOT = c
        break
if IN_COLAB and not (ROOT / "ml").is_dir():
    from google.colab import files
    print("Upload the project ZIP once (AI-Sales-Forecasting-Inventory.zip):")
    for name, data in files.upload().items():
        zipfile.ZipFile(io.BytesIO(data)).extractall("/content")
    ROOT = Path("/content/AI-Sales-Forecasting-Inventory")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["SFIO_PROJECT_ROOT"] = str(ROOT)

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import Image, display, Markdown
from ml import config
config.ensure_directories()
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")
print("Project root:", ROOT.name, "| Colab:", IN_COLAB)

Project root: AI-Sales-Forecasting-Inventory | Colab: False


In [2]:
from ml import forecasting, inventory, analytics
from ml.inventory import InventoryInputs
bundle = forecasting.load_bundle(); panel = forecasting.load_panel(); cal = forecasting.load_calendar()
code = bundle["products"][0]
fc = forecasting.forecast_product(code, 30, bundle, panel, cal)
print(code, "| model:", fc["model_name"], "| origin:", fc["forecast_origin"])
pd.Series(fc["summary"])

85123A | model: hist_gradient_boosting | origin: 2011-12-09 00:00:00


total_forecast_units                        2452.390
average_daily_demand                          81.746
average_demand_per_open_day                   98.095
open_days_in_horizon                          25.000
historical_avg_daily_demand_last_30_days     128.767
historical_total_last_30_days               3863.000
days_since_last_sale                           0.000
dtype: float64

In [3]:
daily = pd.DataFrame(fc["daily"])
hist = pd.DataFrame(fc["history"])
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(pd.to_datetime(hist["date"]), hist["demand"], label="Actual (dataset)")
ax.plot(pd.to_datetime(daily["date"]), daily["predicted_demand"], "--o", ms=3, label="Recursive forecast (model)")
ax.set_title(f"{code}: last 90 days and 30-day forecast"); ax.set_xlabel("Date"); ax.set_ylabel("Units per day"); ax.legend()
plt.show()
for w in fc["warnings"]:
    print("NOTE:", w)

NOTE: Recursive forecast: each day's prediction becomes input for the next day, so errors can accumulate over longer horizons. Treat 30-day forecasts as less certain than 7-day ones.
NOTE: Public holidays after the data end are not known to the model; check forecasts around Christmas/New Year.


## One recommendation (example user inputs)

In [4]:
hist_ext = forecasting.extend_history_to(panel[panel.StockCode == code], pd.Timestamp(fc["forecast_origin"]), cal)
var = inventory.demand_variability(hist_ext)
inp = InventoryInputs(code, current_stock=150, lead_time_days=10, service_level=0.95, review_period_days=7,
                      min_order_qty=100, pack_size=12, max_capacity=2000)   # USER-ENTERED example values
r = inventory.calculate_inventory(inp, daily["predicted_demand"], var, 30)
pd.Series({k: r[k] for k in ["z_value", "average_daily_demand", "demand_std_daily", "safety_stock", "reorder_point",
                             "target_stock", "raw_order_quantity", "suggested_order_quantity", "status"]})

z_value                             1.6449
average_daily_demand                 81.75
demand_std_daily                     160.0
safety_stock                        832.22
reorder_point                      1649.68
target_stock                        2221.9
raw_order_quantity                  2071.9
suggested_order_quantity              1848
status                      Critical Stock
dtype: object

In [5]:
for s in r["order_adjustments"]: print("-", s)
for a in r["alerts"]: print(a["level"].upper(), a["message"])

- Raw need 2071.90 rounded up to 2072 whole units.
- Rounded up from 2072 to 2076 to fit packs of 12.
- Storage capacity 2000 allows only 1848 more units (228 units short of the recommendation).
CRITICAL 85123A: critical stock — order 1848 units immediately.
WARNING Stockout risk: current stock covers less than the 10-day lead-time demand.
WARNING Capacity limit: order reduced by 228 units.


## Sensitivity: service level and lead time
Higher service levels and longer lead times need more safety stock.

In [6]:
rows = []
for sl in (0.90, 0.95, 0.99):
    for lt in (5, 10, 20):
        x = inventory.calculate_inventory(InventoryInputs(code, 150, lt, sl), daily["predicted_demand"], var, 30)
        rows.append({"service_level": sl, "lead_time": lt, "safety_stock": x["safety_stock"], "reorder_point": x["reorder_point"],
                     "order_qty": x["suggested_order_quantity"], "status": x["status"]})
pd.DataFrame(rows)

,service_level,lead_time,safety_stock,reorder_point,order_qty,status
0,0.90,5,458.49,867.22,1290,Critical Stock
1,0.90,10,648.41,1465.87,1889,Critical Stock
2,0.90,20,916.98,2551.91,2975,Critical Stock
3,0.95,5,588.47,997.20,1420,Critical Stock
4,0.95,10,832.22,1649.68,2072,Critical Stock
5,0.95,20,1176.94,2811.86,3235,Critical Stock
6,0.99,5,832.28,1241.01,1664,Critical Stock
7,0.99,10,1177.02,1994.48,2417,Critical Stock
8,0.99,20,1664.56,3299.49,3722,Critical Stock


## ABC analysis (A = first 80% of revenue, B = next 15%, C = last 5%)

In [7]:
from ml.preprocessing import load_clean_sales
abc = analytics.abc_analysis(load_clean_sales())
display(pd.DataFrame(analytics.abc_summary(abc)))
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(abc["revenue_rank"], 100 * abc["cumulative_share"])
ax.axhline(80, ls="--", c="grey"); ax.axhline(95, ls=":", c="grey")
ax.set_title("Cumulative revenue share by product rank (Pareto curve)"); ax.set_xlabel("Products ranked by revenue"); ax.set_ylabel("Cumulative revenue (%)")
plt.show()

,class,products,product_share_pct,revenue,revenue_share_pct
0,A,844,21.62,7861390.29,80.03
1,B,978,25.06,1471407.00,14.98
2,C,2081,53.32,490662.08,4.99
